# Train Model

## A2 Test

In [1]:
# WavLM-Large frozen + layer-weighted head for ComParE 2017 Cold
#
# Split strategy (honest test estimation):
#   train     : ALL of ComParE 'train' split (9505 files)
#   val       : 50% of ComParE 'devel' (stratified) -> early stopping / model selection
#   test      : other 50% of ComParE 'devel' (stratified) -> HONEST held-out read
# Both val and test are speaker-disjoint from train (URTIC guarantees this for
# the official devel/test splits). Random splits of 'train' leak speakers and
# produce val_UAR inflated by ~0.30 over the honest test UAR - do not do that.

import sys
from pathlib import Path
import numpy as np
import torch

MODEL_ROOT = Path.cwd() if Path.cwd().name == "model" else Path("../model")
sys.path.insert(0, str(MODEL_ROOT.resolve()))

from data.data import AudioDataset
from data.cached_dataset import PooledCacheDataset, stratified_split, load_labels
from features.backbone import build_backbone
from features.extract import extract_pooled
from features.head import LayerWeightedPooledHead
from features.train import train_head

# ---------------- config ----------------
DATA_DIR      = "../dataset/ComParE2017_Cold_4students"
CACHE_ROOT    = "../cache"
BACKBONE      = "wavlm-large"          # "wavlm-base-plus" for fast debug
DEVICE        = "cuda" if torch.cuda.is_available() else "cpu"
CLIP_SECS     = 8.0
BATCH_EXTRACT = 4                      # WavLM-Large fp16 on a 4090
BATCH_TRAIN   = 64
EPOCHS        = 25
PATIENCE      = 6
BASE_LR       = 1e-3
WEIGHT_DECAY  = 1e-3
PROJ_DIM      = 128                    # smaller than before to combat overfitting
DROPOUT       = 0.5
SEED          = 42
DEVEL_VAL_FRAC = 0.5                   # split devel 50/50 for val/test

# ---------------- 1. extract pooled features (idempotent) ----------------
backbone = build_backbone(BACKBONE, device=DEVICE)
print(f"backbone={BACKBONE}  n_layers={backbone.n_layers}  hidden_dim={backbone.hidden_dim}")

for split in ("train", "devel"):
    ds = AudioDataset(
        data_dir=DATA_DIR, split=split,
        use_mel=False, use_opensmile=False,
        pad_or_truncate_secs=CLIP_SECS,
    )
    print(f"extract[{split}] n={len(ds)}")
    manifest = extract_pooled(
        backbone=backbone, dataset=ds,
        cache_root=CACHE_ROOT,
        batch_size=BATCH_EXTRACT,
        skip_existing=True,
    )
    print(f"  wrote {manifest.n_chunks} new chunks  stat_dim={manifest.stat_dim}")

del backbone
if DEVICE == "cuda":
    torch.cuda.empty_cache()

# ---------------- 2. datasets ----------------
backbone_id = f"microsoft_{BACKBONE}"

train_ds   = PooledCacheDataset(DATA_DIR, CACHE_ROOT, backbone_id, split="train")
full_devel = PooledCacheDataset(DATA_DIR, CACHE_ROOT, backbone_id, split="devel")

labels_map = load_labels(DATA_DIR)
val_files, test_files = stratified_split(full_devel.files, labels_map,
                                         val_frac=DEVEL_VAL_FRAC, seed=SEED)
val_ds  = PooledCacheDataset(DATA_DIR, CACHE_ROOT, backbone_id, file_list=val_files)
test_ds = PooledCacheDataset(DATA_DIR, CACHE_ROOT, backbone_id, file_list=test_files)

print(f"train  n={len(train_ds):5d}  counts={train_ds.class_counts()}")
print(f"val    n={len(val_ds):5d}  counts={val_ds.class_counts()}  (devel half, speaker-disjoint)")
print(f"test   n={len(test_ds):5d}  counts={test_ds.class_counts()}  (devel half, speaker-disjoint, held-out)")

# ---------------- 3. head + train ----------------
sample = train_ds[0]["pooled"]
n_layers, stat_dim = sample.shape

head = LayerWeightedPooledHead(
    n_layers=n_layers, stat_dim=stat_dim,
    proj_dim=PROJ_DIM, n_classes=2, dropout=DROPOUT,
)

result = train_head(
    head=head,
    train_ds=train_ds,
    val_ds=val_ds,
    test_ds=test_ds,                    # speaker-disjoint, touched once
    epochs=EPOCHS,
    batch_size=BATCH_TRAIN,
    base_lr=BASE_LR,
    weight_decay=WEIGHT_DECAY,
    early_stop_patience=PATIENCE,
    class_weights=None,                 # balanced_sampler handles it
    balanced_sampler=True,              # ~50/50 C/NC per batch via WeightedRandomSampler
    fit_scaler=True,                    # z-score each of the 25*4096 feature positions
    device=DEVICE,
    ckpt_path=f"../checkpoints/{BACKBONE}_head_best.pt",
    seed=SEED,
)

# ---------------- 4. summary ----------------
print("\n=== SUMMARY ===")
print(f"best val_UAR    = {result.best_val_uar:.4f}  (epoch {result.best_epoch})")
print(f"test UAR        = {result.test_uar:.4f}   <-- honest estimate for hidden test")
print(f"test accuracy   = {result.test_acc:.4f}")
print(f"test recall     = C:{result.test_per_class_recall.get(1, 0):.4f}  "
      f"NC:{result.test_per_class_recall.get(0, 0):.4f}")
top3 = np.argsort(result.layer_weights)[::-1][:3]
print(f"dominant layers : " + ", ".join(
    f"L{int(i)}={result.layer_weights[i]:.3f}" for i in top3))


backbone=wavlm-large  n_layers=25  hidden_dim=1024
extract[train] n=9505


extract[microsoft_wavlm-large]:   0%|          | 0/2377 [00:00<?, ?it/s]

  wrote 0 new chunks  stat_dim=4096
extract[devel] n=9596


extract[microsoft_wavlm-large]:   0%|          | 0/2399 [00:00<?, ?it/s]

  wrote 0 new chunks  stat_dim=4096
train  n= 9505  counts={1: 970, 0: 8535}
val    n= 4798  counts={0: 4293, 1: 505}  (devel half, speaker-disjoint)
test   n= 4798  counts={0: 4292, 1: 506}  (devel half, speaker-disjoint, held-out)


...

[epoch 09] train_loss=0.0637 train_acc=0.976  |  val_UAR=0.6316 val_acc=0.857 (C=0.347 NC=0.917)  |  top_layers=[L4:0.04, L3:0.04, L5:0.04]


ep 10/25:   0%|          | 0/149 [00:00<?, ?it/s]

[epoch 10] train_loss=0.0760 train_acc=0.972  |  val_UAR=0.6061 val_acc=0.881 (C=0.257 NC=0.955)  |  top_layers=[L4:0.04, L3:0.04, L2:0.04]


ep 11/25:   0%|          | 0/149 [00:00<?, ?it/s]

[epoch 11] train_loss=0.0467 train_acc=0.985  |  val_UAR=0.6166 val_acc=0.878 (C=0.285 NC=0.948)  |  top_layers=[L4:0.04, L2:0.04, L3:0.04]

[train] early stop at epoch 11 (no val_UAR improvement for 6 epochs)

[train] best val_UAR=0.6346 at epoch 5

[HELD-OUT TEST] devel (speaker-disjoint from train, proxy for hidden test):
    UAR       = 0.6287
    accuracy  = 0.7787
    recall_C  = 0.4387
    recall_NC = 0.8187
    val-to-test gap = +0.0059  (consistent)

=== SUMMARY ===
best val_UAR    = 0.6346  (epoch 5)
test UAR        = 0.6287   <-- honest estimate for hidden test
test accuracy   = 0.7787
test recall     = C:0.4387  NC:0.8187
dominant layers : L4=0.042, L2=0.042, L1=0.042


## A2 Baseline 

In [1]:
import json
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader

from data.cached_dataset import PooledCacheDataset, stratified_split, load_labels
from features import (
    LayerWeightedPooledHead,
    train_head,
    evaluate,
    sweep_threshold,
    evaluate_at_threshold,
)
from features.train import _pooled_collate

DATA_DIR   = "../dataset/ComParE2017_Cold_4students"
CACHE_ROOT = "../cache"
BACKBONE   = "microsoft_wavlm-large"
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
SEED       = 42

# --- splits ---
full_train = PooledCacheDataset(DATA_DIR, CACHE_ROOT, BACKBONE, split="train")
full_devel = PooledCacheDataset(DATA_DIR, CACHE_ROOT, BACKBONE, split="devel")
labels_map = load_labels(DATA_DIR)

train_fit_files, train_thr_files   = stratified_split(full_train.files, labels_map, val_frac=0.10, seed=SEED)
devel_val_files, devel_test_files  = stratified_split(full_devel.files, labels_map, val_frac=0.50, seed=SEED)

train_fit_ds   = PooledCacheDataset(DATA_DIR, CACHE_ROOT, BACKBONE, file_list=train_fit_files)
train_thr_ds   = PooledCacheDataset(DATA_DIR, CACHE_ROOT, BACKBONE, file_list=train_thr_files)
devel_val_ds   = PooledCacheDataset(DATA_DIR, CACHE_ROOT, BACKBONE, file_list=devel_val_files)
devel_test_ds  = PooledCacheDataset(DATA_DIR, CACHE_ROOT, BACKBONE, file_list=devel_test_files)

print(f"train_fit={len(train_fit_ds)}  train_threshold={len(train_thr_ds)}  "
      f"devel_val={len(devel_val_ds)}  devel_test={len(devel_test_ds)}")

# --- head ---
sample = full_train[0]["pooled"]
n_layers, stat_dim = sample.shape
head = LayerWeightedPooledHead(n_layers=n_layers, stat_dim=stat_dim,
                               proj_dim=128, n_classes=2, dropout=0.5)

# --- train (A2 frozen recipe) ---
result = train_head(
    head, train_fit_ds, devel_val_ds, test_ds=devel_test_ds,
    epochs=25, batch_size=64, base_lr=1e-3, weight_decay=1e-4,
    early_stop_patience=6, class_weights=None, balanced_sampler=True,
    fit_scaler=True, device=DEVICE,
    ckpt_path=f"../cache/{BACKBONE}/head_A2.pt",
    num_workers=0, seed=SEED,
)

# --- threshold calibration on train_threshold (NOT devel) ---
thr_loader = DataLoader(train_thr_ds, batch_size=256, shuffle=False,
                        collate_fn=_pooled_collate, num_workers=0)
tau, tau_uar_trainthr, _ = sweep_threshold(head, thr_loader, DEVICE)
print(f"\n[calib] tau*={tau:.3f}  UAR on train_threshold @ tau*={tau_uar_trainthr:.4f}")

# --- honest eval on devel_test at both argmax and tau ---
test_loader = DataLoader(devel_test_ds, batch_size=256, shuffle=False,
                         collate_fn=_pooled_collate, num_workers=0)
uar_arg, acc_arg, pcr_arg, _, _ = evaluate(head, test_loader, DEVICE)
uar_tau, acc_tau, pcr_tau       = evaluate_at_threshold(head, test_loader, DEVICE, tau)

print(f"\n[devel_test @ argmax]    UAR={uar_arg:.4f}  acc={acc_arg:.4f}  "
      f"C={pcr_arg.get(1,0):.4f}  NC={pcr_arg.get(0,0):.4f}")
print(f"[devel_test @ tau={tau:.2f}] UAR={uar_tau:.4f}  acc={acc_tau:.4f}  "
      f"C={pcr_tau.get(1,0):.4f}  NC={pcr_tau.get(0,0):.4f}")

# --- write results/A2.json ---
out = {
    "rung_id": "A2",
    "description": "Frozen WavLM-Large, 25 hidden states, pooled mean+std+skew+kurt, FeatureStandardiser, softmax layer weights (lr x0.1), 2-layer MLP + BatchNorm, balanced sampler, no class weights.",
    "backbone": "microsoft/wavlm-large",
    "seed": SEED,
    "split": {
        "train_fit": len(train_fit_ds), "train_threshold": len(train_thr_ds),
        "devel_val": len(devel_val_ds), "devel_test": len(devel_test_ds),
    },
    "best_epoch": result.best_epoch,
    "best_val_uar": float(result.best_val_uar),
    "tau_calibrated": float(tau),
    "uar_train_threshold_at_tau": float(tau_uar_trainthr),
    "devel_test_argmax":     {"uar": float(uar_arg), "acc": float(acc_arg),
                              "recall_C": float(pcr_arg.get(1,0)), "recall_NC": float(pcr_arg.get(0,0))},
    "devel_test_calibrated": {"uar": float(uar_tau), "acc": float(acc_tau),
                              "recall_C": float(pcr_tau.get(1,0)), "recall_NC": float(pcr_tau.get(0,0))},
    "val_test_gap": float(result.best_val_uar - uar_arg),
    "layer_weights": [float(x) for x in result.layer_weights.tolist()],
    "speaker_probe_top1": None,
    "speaker_probe_nmi": None,
}
Path("../results").mkdir(exist_ok=True)
Path("../results/A2.json").write_text(json.dumps(out, indent=2))
print("\n[wrote] results/A2.json")


train_fit=8554  train_threshold=951  devel_val=4798  devel_test=4798


fit_scaler:   0%|          | 0/34 [00:00<?, ?it/s]

[scaler] fit on n=8554  mean_abs_avg=2.3363  std range=[1.2663e-03, 4.9854e+01]  ratio=3.94e+04
[train] device=cuda  train=8554  val=4798  test=4798
[train] class_weights=None
[train] balanced_sampler=True  fit_scaler=True
[train] epochs=25  batch=64  lr=0.001  patience=6
[train] head params: n_layers=25  stat_dim=4096  proj_dim=128
[train] majority-class baseline UAR = 0.500 by definition
[diag] untrained batch: logit_range=[-1.508, +2.794]  mean_p_C=0.541  loss=0.7529  any_nan=False  any_inf=False



ep 01/25:   0%|          | 0/134 [00:00<?, ?it/s]

[epoch 01] train_loss=0.5212 train_acc=0.726  |  val_UAR=0.6299 val_acc=0.764 (C=0.459 NC=0.800)  |  top_layers=[L1:0.04, L2:0.04, L4:0.04]  *


...

[epoch 07] train_loss=0.0958 train_acc=0.967  |  val_UAR=0.6320 val_acc=0.854 (C=0.350 NC=0.914)  |  top_layers=[L4:0.04, L5:0.04, L3:0.04]


ep 08/25:   0%|          | 0/134 [00:00<?, ?it/s]

[epoch 08] train_loss=0.0983 train_acc=0.964  |  val_UAR=0.6324 val_acc=0.864 (C=0.339 NC=0.926)  |  top_layers=[L4:0.04, L5:0.04, L6:0.04]


ep 09/25:   0%|          | 0/134 [00:00<?, ?it/s]

[epoch 09] train_loss=0.0846 train_acc=0.968  |  val_UAR=0.6302 val_acc=0.875 (C=0.321 NC=0.940)  |  top_layers=[L4:0.04, L5:0.04, L3:0.04]

[train] early stop at epoch 9 (no val_UAR improvement for 6 epochs)

[train] best val_UAR=0.6478 at epoch 3

[HELD-OUT TEST] devel (speaker-disjoint from train, proxy for hidden test):
    UAR       = 0.6510
    accuracy  = 0.8153
    recall_C  = 0.4427
    recall_NC = 0.8593
    val-to-test gap = -0.0032  (consistent)

[calib] tau*=0.495  UAR on train_threshold @ tau*=0.9383

[devel_test @ argmax]    UAR=0.6510  acc=0.8153  C=0.4427  NC=0.8593
[devel_test @ tau=0.49] UAR=0.6513  acc=0.8143  C=0.4447  NC=0.8579

[wrote] results/A2.json


## Improve Baseline

In [2]:
import json
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader

from data.cached_dataset import PooledCacheDataset, stratified_split, load_labels
from features import (
    LayerWeightedPooledHead,
    train_head,
    evaluate,
    sweep_threshold,
    evaluate_at_threshold,
)
from features.train import _pooled_collate

DATA_DIR   = "../dataset/ComParE2017_Cold_4students"
CACHE_ROOT = "../cache"
BACKBONE   = "microsoft_wavlm-large"
DEVICE     = "cuda" if torch.cuda.is_available() else "cpu"
SPLIT_SEED = 42
SEEDS      = [42, 123, 7]

# --- splits (fixed across all training seeds) ---
full_train = PooledCacheDataset(DATA_DIR, CACHE_ROOT, BACKBONE, split="train")
full_devel = PooledCacheDataset(DATA_DIR, CACHE_ROOT, BACKBONE, split="devel")
labels_map = load_labels(DATA_DIR)

train_fit_files, train_thr_files  = stratified_split(full_train.files, labels_map, val_frac=0.10, seed=SPLIT_SEED)
devel_val_files, devel_test_files = stratified_split(full_devel.files, labels_map, val_frac=0.50, seed=SPLIT_SEED)

train_fit_ds  = PooledCacheDataset(DATA_DIR, CACHE_ROOT, BACKBONE, file_list=train_fit_files)
train_thr_ds  = PooledCacheDataset(DATA_DIR, CACHE_ROOT, BACKBONE, file_list=train_thr_files)
devel_val_ds  = PooledCacheDataset(DATA_DIR, CACHE_ROOT, BACKBONE, file_list=devel_val_files)
devel_test_ds = PooledCacheDataset(DATA_DIR, CACHE_ROOT, BACKBONE, file_list=devel_test_files)

print(f"train_fit={len(train_fit_ds)}  train_threshold={len(train_thr_ds)}  "
      f"devel_val={len(devel_val_ds)}  devel_test={len(devel_test_ds)}\n")

sample = full_train[0]["pooled"]
n_layers, stat_dim = sample.shape

thr_loader  = DataLoader(train_thr_ds,  batch_size=256, shuffle=False, collate_fn=_pooled_collate)
test_loader = DataLoader(devel_test_ds, batch_size=256, shuffle=False, collate_fn=_pooled_collate)

# --- seed loop ---
runs = []
for seed in SEEDS:
    print(f"\n{'=' * 70}\nSEED {seed}\n{'=' * 70}")
    head = LayerWeightedPooledHead(n_layers=n_layers, stat_dim=stat_dim,
                                   proj_dim=128, n_classes=2, dropout=0.5)
    result = train_head(
        head, train_fit_ds, devel_val_ds, test_ds=devel_test_ds,
        epochs=25, batch_size=64, base_lr=1e-3, weight_decay=1e-4,
        early_stop_patience=6, class_weights=None, balanced_sampler=True,
        fit_scaler=True, device=DEVICE,
        ckpt_path=f"../cache/{BACKBONE}/head_A2_seed{seed}.pt",
        num_workers=0, seed=seed,
    )
    tau, tau_uar_trainthr, _ = sweep_threshold(head, thr_loader, DEVICE)
    uar_arg, acc_arg, pcr_arg, _, _ = evaluate(head, test_loader, DEVICE)
    uar_tau, acc_tau, pcr_tau       = evaluate_at_threshold(head, test_loader, DEVICE, tau)
    print(f"\n[seed {seed}] tau*={tau:.3f}  devel_test UAR: argmax={uar_arg:.4f}  calib={uar_tau:.4f}  "
          f"C={pcr_tau.get(1,0):.3f} NC={pcr_tau.get(0,0):.3f}")
    runs.append({
        "seed": seed,
        "best_epoch": result.best_epoch,
        "best_val_uar": float(result.best_val_uar),
        "tau": float(tau),
        "uar_train_threshold_at_tau": float(tau_uar_trainthr),
        "devel_test_argmax":     {"uar": float(uar_arg), "acc": float(acc_arg),
                                  "recall_C": float(pcr_arg.get(1,0)), "recall_NC": float(pcr_arg.get(0,0))},
        "devel_test_calibrated": {"uar": float(uar_tau), "acc": float(acc_tau),
                                  "recall_C": float(pcr_tau.get(1,0)), "recall_NC": float(pcr_tau.get(0,0))},
        "val_test_gap": float(result.best_val_uar - uar_arg),
        "layer_weights": [float(x) for x in result.layer_weights.tolist()],
    })

# --- aggregate ---
def agg(key_path):
    vals = np.array([eval("r" + "".join(f"[{k!r}]" for k in key_path)) for r in runs])
    return float(vals.mean()), float(vals.std(ddof=1) if len(vals) > 1 else 0.0)

uar_arg_mean, uar_arg_std = agg(["devel_test_argmax", "uar"])
uar_tau_mean, uar_tau_std = agg(["devel_test_calibrated", "uar"])
rc_mean, rc_std           = agg(["devel_test_calibrated", "recall_C"])
rnc_mean, rnc_std         = agg(["devel_test_calibrated", "recall_NC"])
gap_mean, gap_std         = agg(["val_test_gap"])

print(f"\n{'=' * 70}\nA2 DISTRIBUTION (N={len(SEEDS)} seeds, splits fixed)\n{'=' * 70}")
print(f"  UAR argmax     : {uar_arg_mean:.4f} +/- {uar_arg_std:.4f}")
print(f"  UAR calibrated : {uar_tau_mean:.4f} +/- {uar_tau_std:.4f}")
print(f"  recall_C       : {rc_mean:.4f} +/- {rc_std:.4f}")
print(f"  recall_NC      : {rnc_mean:.4f} +/- {rnc_std:.4f}")
print(f"  val-test gap   : {gap_mean:+.4f} +/- {gap_std:.4f}")

out = {
    "rung_id": "A2",
    "description": "Frozen WavLM-Large, 25 hidden states, pooled mean+std+skew+kurt, FeatureStandardiser, softmax layer weights (lr x0.1), 2-layer MLP + BatchNorm, balanced sampler, no class weights.",
    "backbone": "microsoft/wavlm-large",
    "split_seed": SPLIT_SEED,
    "training_seeds": SEEDS,
    "split_sizes": {
        "train_fit": len(train_fit_ds), "train_threshold": len(train_thr_ds),
        "devel_val": len(devel_val_ds), "devel_test": len(devel_test_ds),
    },
    "distribution": {
        "uar_argmax":     {"mean": uar_arg_mean, "std": uar_arg_std, "n": len(SEEDS)},
        "uar_calibrated": {"mean": uar_tau_mean, "std": uar_tau_std, "n": len(SEEDS)},
        "recall_C":       {"mean": rc_mean,      "std": rc_std,      "n": len(SEEDS)},
        "recall_NC":      {"mean": rnc_mean,     "std": rnc_std,     "n": len(SEEDS)},
        "val_test_gap":   {"mean": gap_mean,     "std": gap_std,     "n": len(SEEDS)},
    },
    "runs": runs,
    "speaker_probe_top1": None,
    "speaker_probe_nmi": None,
}
Path("../results").mkdir(exist_ok=True)
Path("../results/A2.json").write_text(json.dumps(out, indent=2))
print("\n[wrote] results/A2.json (distribution over 3 seeds)")


train_fit=8554  train_threshold=951  devel_val=4798  devel_test=4798


SEED 42


fit_scaler:   0%|          | 0/34 [00:00<?, ?it/s]

[scaler] fit on n=8554  mean_abs_avg=2.3363  std range=[1.2663e-03, 4.9854e+01]  ratio=3.94e+04
[train] device=cuda  train=8554  val=4798  test=4798
[train] class_weights=None
[train] balanced_sampler=True  fit_scaler=True
[train] epochs=25  batch=64  lr=0.001  patience=6
[train] head params: n_layers=25  stat_dim=4096  proj_dim=128
[train] majority-class baseline UAR = 0.500 by definition
[diag] untrained batch: logit_range=[-1.561, +1.462]  mean_p_C=0.547  loss=0.8351  any_nan=False  any_inf=False



ep 01/25:   0%|          | 0/134 [00:00<?, ?it/s]

[epoch 01] train_loss=0.5264 train_acc=0.732  |  val_UAR=0.6202 val_acc=0.825 (C=0.360 NC=0.880)  |  top_layers=[L1:0.04, L4:0.04, L2:0.04]  *


...

[epoch 06] train_loss=0.1144 train_acc=0.958  |  val_UAR=0.6432 val_acc=0.856 (C=0.374 NC=0.912)  |  top_layers=[L4:0.04, L6:0.04, L1:0.04]


ep 07/25:   0%|          | 0/134 [00:00<?, ?it/s]

[epoch 07] train_loss=0.1013 train_acc=0.962  |  val_UAR=0.6203 val_acc=0.861 (C=0.315 NC=0.926)  |  top_layers=[L4:0.04, L6:0.04, L5:0.04]


ep 08/25:   0%|          | 0/134 [00:00<?, ?it/s]

[epoch 08] train_loss=0.0868 train_acc=0.967  |  val_UAR=0.6057 val_acc=0.881 (C=0.257 NC=0.954)  |  top_layers=[L4:0.04, L6:0.04, L5:0.04]

[train] early stop at epoch 8 (no val_UAR improvement for 6 epochs)

[train] best val_UAR=0.6467 at epoch 2

[HELD-OUT TEST] devel (speaker-disjoint from train, proxy for hidden test):
    UAR       = 0.6433
    accuracy  = 0.8406
    recall_C  = 0.3933
    recall_NC = 0.8933
    val-to-test gap = +0.0034  (consistent)

[seed 7] tau*=0.470  devel_test UAR: argmax=0.6433  calib=0.6466  C=0.411 NC=0.882

A2 DISTRIBUTION (N=3 seeds, splits fixed)
  UAR argmax     : 0.6428 +/- 0.0034
  UAR calibrated : 0.6464 +/- 0.0082
  recall_C       : 0.4321 +/- 0.0284
  recall_NC      : 0.8607 +/- 0.0192
  val-test gap   : -0.0009 +/- 0.0047

[wrote] results/A2.json (distribution over 3 seeds)


## ECAPA embeddings extraction

In [1]:
from pathlib import Path
import torch

from speakers import extract_ecapa

DATA_DIR   = Path("../dataset/ComParE2017_Cold_4students")
CACHE_ROOT = Path("../cache")
OUT_DIR    = CACHE_ROOT / "ecapa-voxceleb"
DEVICE     = "cuda" if torch.cuda.is_available() else "mps" if torch.mps.is_available() else "cpu"

print(f"device: {DEVICE}")

wav_dir   = DATA_DIR / "wav"
wav_paths = sorted(wav_dir.glob("*.wav"))

# split into train/devel/test purely for reporting; extraction is split-agnostic
splits = {
    "train": [p for p in wav_paths if p.name.startswith("train_")],
    "devel": [p for p in wav_paths if p.name.startswith("devel_")],
    "test":  [p for p in wav_paths if p.name.startswith("test_")],
}
for k, v in splits.items():
    print(f"{k:6s}: {len(v)} files")
print(f"total : {len(wav_paths)} files")

n_new = extract_ecapa(
    wav_paths=wav_paths,
    out_dir=OUT_DIR,
    device=DEVICE,
    batch_size=16,
    max_seconds=30.0,
    skip_existing=True,
    num_workers=8
)
print(f"\n[ecapa] newly extracted: {n_new}")
print(f"[ecapa] cache location   : {OUT_DIR.resolve()}")

# sanity: load one and inspect
sample = sorted(OUT_DIR.glob("*.pt"))[0]
emb = torch.load(sample, weights_only=True)
print(f"\n[sanity] {sample.name}: shape={tuple(emb.shape)} dtype={emb.dtype} "
      f"norm={emb.float().norm().item():.3f}")


device: mps
train : 9505 files
devel : 9596 files
test  : 9551 files
total : 28652 files
[ecapa] to_process=24236  skipped_existing=4416  out=../cache/ecapa-voxceleb


/opt/homebrew/Caskroom/miniforge/base/envs/thesis/lib/python3.11/site-packages/speechbrain/utils/checkpoints.py:202: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict

ecapa:   0%|          | 0/1515 [00:18<?, ?it/s]


[ecapa] newly extracted: 24236
[ecapa] cache location   : /Users/chris/Desktop/Research/advanced_ai_for_health/AI-For-Health/cache/ecapa-voxceleb

[sanity] devel_0001.pt: shape=(192,) dtype=torch.float16 norm=316.415


## Clustering

In [1]:
from pathlib import Path
from speakers import fit_and_assign

DATA_DIR    = Path("../dataset/ComParE2017_Cold_4students")
ECAPA_CACHE = Path("../cache/ecapa-voxceleb")
PSEUDO_OUT  = Path("../cache/pseudo_speakers")
SEED        = 42

wav_dir     = DATA_DIR / "wav"
train_paths = sorted(wav_dir.glob("train_*.wav"))
devel_paths = sorted(wav_dir.glob("devel_*.wav"))
test_paths  = sorted(wav_dir.glob("test_*.wav"))
print(f"train={len(train_paths)}  devel={len(devel_paths)}  test={len(test_paths)}")

reports = fit_and_assign(
    train_paths=train_paths,
    devel_paths=devel_paths,
    test_paths=test_paths,
    ecapa_cache=ECAPA_CACHE,
    out_dir=PSEUDO_OUT,
    ks=(100, 210, 420),
    seed=SEED,
    silhouette_sample=3000,
)

print("\n" + "=" * 74)
print(f"K-SWEEP SUMMARY  (higher silhouette = better; lower intra/inter = tighter)")
print("=" * 74)
print(f"{'k':>4}  {'silhouette':>11}  {'intra/inter':>12}  "
      f"{'size min/med/max':>18}  {'unused@devel':>14}")
for r in reports:
    print(f"{r.k:>4}  {r.silhouette:>11.4f}  {r.intra_inter_ratio:>12.4f}  "
          f"{r.cluster_size_min:>5}/{r.cluster_size_median:<4}/{r.cluster_size_max:<5}  "
          f"{r.n_empty_on_devel:>5} / {r.k}")


train=9505  devel=9596  test=9551
[cluster] loading cached ECAPA embeddings ...
  train: (9505, 192)   devel: (9596, 192)   test: (9551, 192)

[cluster] k=100  (KMeans, n_init=10) ...
  silhouette (n=3000) : 0.1917
  intra/inter ratio        : 0.6555
  cluster sizes (train)    : min=31  median=83  max=210  mean=95.0
  clusters unused on devel : 2 / 100
  wrote ..\cache\pseudo_speakers\k100_seed42.tsv

[cluster] k=210  (KMeans, n_init=10) ...
  silhouette (n=3000) : 0.2311
  intra/inter ratio        : 0.5108
  cluster sizes (train)    : min=3  median=45  max=104  mean=45.3
  clusters unused on devel : 4 / 210
  wrote ..\cache\pseudo_speakers\k210_seed42.tsv

[cluster] k=420  (KMeans, n_init=10) ...
  silhouette (n=3000) : 0.1035
  intra/inter ratio        : 0.4643
  cluster sizes (train)    : min=1  median=21  max=74  mean=22.6
  clusters unused on devel : 27 / 420
  wrote ..\cache\pseudo_speakers\k420_seed42.tsv

K-SWEEP SUMMARY  (higher silhouette = better; lower intra/inter = tighter

In [1]:
import json
from pathlib import Path

import numpy as np
import torch

from data.cached_dataset import PooledCacheDataset, stratified_split, load_labels
from features import LayerWeightedPooledHead
from speakers import load_pseudo_speakers, extract_z, train_probe
from speakers.probe import _align_labels

DATA_DIR    = "../dataset/ComParE2017_Cold_4students"
CACHE_ROOT  = "../cache"
BACKBONE    = "microsoft_wavlm-large"
PSEUDO_TSV  = Path("../cache/pseudo_speakers/k210_seed42.tsv")
RESULTS_PATH = Path("../results/A2.json")
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
SEEDS       = [42, 123, 7]
K_CLUSTERS  = 210
SPLIT_SEED  = 42

# --- datasets: same splits as A2 training ---
full_train = PooledCacheDataset(DATA_DIR, CACHE_ROOT, BACKBONE, split="train")
full_devel = PooledCacheDataset(DATA_DIR, CACHE_ROOT, BACKBONE, split="devel")
labels_map = load_labels(DATA_DIR)

train_fit_files, _ = stratified_split(full_train.files, labels_map, val_frac=0.10, seed=SPLIT_SEED)
train_fit_ds = PooledCacheDataset(DATA_DIR, CACHE_ROOT, BACKBONE, file_list=train_fit_files)

# --- pseudo-speaker assignments ---
assignments = load_pseudo_speakers(PSEUDO_TSV)
print(f"[probe] loaded k=210 assignments for {len(assignments)} files from {PSEUDO_TSV.name}")

# --- per-checkpoint probe ---
sample = full_train[0]["pooled"]
n_layers, stat_dim = sample.shape

results = []
for seed in SEEDS:
    print(f"\n{'=' * 70}\nPROBE on A2 seed={seed}\n{'=' * 70}")
    ckpt_path = Path(f"../cache/{BACKBONE}/head_A2_seed{seed}.pt")
    ckpt = torch.load(ckpt_path, weights_only=True, map_location=DEVICE)
    head = LayerWeightedPooledHead(
        n_layers=ckpt["n_layers"], stat_dim=ckpt["stat_dim"],
        proj_dim=ckpt["proj_dim"], n_classes=2,
    )
    head.load_state_dict(ckpt["state_dict"])

    print(f"  extracting z ...")
    z_train, names_train = extract_z(head, train_fit_ds, DEVICE)
    z_devel, names_devel = extract_z(head, full_devel,    DEVICE)
    y_train = _align_labels(names_train, assignments)
    y_devel = _align_labels(names_devel, assignments)
    print(f"  z_train={tuple(z_train.shape)}  z_devel={tuple(z_devel.shape)}  "
          f"y unique (train/devel)={len(np.unique(y_train))}/{len(np.unique(y_devel))}")

    r = train_probe(
        z_train, y_train, z_devel, y_devel,
        n_clusters=K_CLUSTERS, device=DEVICE, seed=seed,
        epochs=50, batch_size=256, lr=1e-3, hidden_dim=256, dropout=0.1,
    )
    chance = 1.0 / K_CLUSTERS
    print(f"  [probe seed {seed}] top1={r.top1:.4f} (chance={chance:.4f}, "
          f"{r.top1/chance:.1f}× above chance)  NMI={r.nmi:.4f}  "
          f"train_top1={r.train_top1:.4f}  best@ep{r.best_eval_top1_epoch}")
    results.append({"seed": seed, "top1": r.top1, "nmi": r.nmi,
                    "train_top1": r.train_top1, "best_epoch": r.best_eval_top1_epoch})

top1s = np.array([r["top1"] for r in results])
nmis  = np.array([r["nmi"]  for r in results])
chance = 1.0 / K_CLUSTERS

print(f"\n{'=' * 70}\nA2 SPEAKER PROBE DISTRIBUTION (k={K_CLUSTERS}, N={len(SEEDS)})\n{'=' * 70}")
print(f"  top1 : {top1s.mean():.4f} +/- {top1s.std(ddof=1):.4f}  "
      f"(chance={chance:.4f}, {top1s.mean()/chance:.1f}x above chance)")
print(f"  NMI  : {nmis.mean():.4f} +/- {nmis.std(ddof=1):.4f}")

# --- patch results/A2.json ---
a2 = json.loads(RESULTS_PATH.read_text())
a2["speaker_probe"] = {
    "k": K_CLUSTERS,
    "pseudo_speaker_tsv": PSEUDO_TSV.name,
    "chance_top1": chance,
    "top1":        {"mean": float(top1s.mean()), "std": float(top1s.std(ddof=1)), "n": len(SEEDS)},
    "nmi":         {"mean": float(nmis.mean()),  "std": float(nmis.std(ddof=1)),  "n": len(SEEDS)},
    "runs":        results,
}
a2["speaker_probe_top1"] = a2["speaker_probe"]["top1"]
a2["speaker_probe_nmi"]  = a2["speaker_probe"]["nmi"]
RESULTS_PATH.write_text(json.dumps(a2, indent=2))
print(f"\n[wrote] {RESULTS_PATH}  (added speaker_probe block)")


[probe] loaded k=210 assignments for 28652 files from k210_seed42.tsv

PROBE on A2 seed=42
  extracting z ...
  z_train=(8554, 128)  z_devel=(9596, 128)  y unique (train/devel)=210/206
  [probe ep 01/50] loss=4.9296  train_top1=0.0445  eval_top1=0.0147  best=0.0147@1
  [probe ep 10/50] loss=1.4069  train_top1=0.6547  eval_top1=0.0424  best=0.0440@8
  [probe ep 20/50] loss=0.7421  train_top1=0.8062  eval_top1=0.0466  best=0.0484@19
  [probe ep 30/50] loss=0.4950  train_top1=0.8659  eval_top1=0.0459  best=0.0484@19
  [probe ep 40/50] loss=0.3725  train_top1=0.9007  eval_top1=0.0429  best=0.0492@32
  [probe ep 50/50] loss=0.2827  train_top1=0.9258  eval_top1=0.0484  best=0.0492@32
  [probe seed 42] top1=0.0492 (chance=0.0048, 10.3× above chance)  NMI=0.3738  train_top1=0.9258  best@ep32

PROBE on A2 seed=123
  extracting z ...
  z_train=(8554, 128)  z_devel=(9596, 128)  y unique (train/devel)=210/206
  [probe ep 01/50] loss=4.8547  train_top1=0.0516  eval_top1=0.0194  best=0.0194@1
  [pro

## A3 — Frame-level cache (WavLM layers {1, 4, 8, 12, 16, 20, 24})

Per-frame hidden states at 50 Hz, stripped of padding, one file per (layer, utterance):
`cache/microsoft_wavlm-large/frames/L{N}/{stem}.pt`  shape `[T_valid, 1024]` fp16.

Per-layer file layout lets A3/A4/A6 load only the layers they need. Idempotent via `skip_existing=True`.

In [ ]:
# A3 frame-cache extraction - run once, idempotent.
# Self-contained: does not depend on the A2 config cell.
import sys, torch
from pathlib import Path

MODEL_ROOT = Path.cwd() if Path.cwd().name == 'model' else Path('../model')
if str(MODEL_ROOT.resolve()) not in sys.path:
    sys.path.insert(0, str(MODEL_ROOT.resolve()))

from features.backbone import build_backbone
from features.extract import extract_frames, DEFAULT_FRAME_LAYERS
from data.data import AudioDataset

# ---- config (mirrors A2 cell) ----
DATA_DIR      = '../dataset/ComParE2017_Cold_4students'
CACHE_ROOT    = '../cache'
BACKBONE      = 'wavlm-large'
DEVICE        = 'cuda' if torch.cuda.is_available() else 'cpu'
CLIP_SECS     = 8.0
BATCH_EXTRACT = 4
FRAME_LAYERS  = DEFAULT_FRAME_LAYERS   # (1, 4, 8, 12, 16, 20, 24)

backbone = build_backbone(BACKBONE, device=DEVICE)
print(f'frames[{BACKBONE}] layers={FRAME_LAYERS}')

for split in ('train', 'devel'):
    ds = AudioDataset(
        data_dir=DATA_DIR, split=split,
        use_mel=False, use_opensmile=False,
        pad_or_truncate_secs=CLIP_SECS,
    )
    print(f'frames[{split}] n={len(ds)}')
    manifest = extract_frames(
        backbone=backbone, dataset=ds,
        cache_root=CACHE_ROOT,
        layers=FRAME_LAYERS,
        batch_size=BATCH_EXTRACT,
        skip_existing=True,
    )
    print(f'  wrote {manifest.n_chunks} new (layer, file) tensors')

del backbone
if DEVICE == 'cuda':
    torch.cuda.empty_cache()

# --- shape spot-check on one stem ---
frames_root = Path(CACHE_ROOT) / f'microsoft_{BACKBONE}' / 'frames'
any_stem = next(iter((frames_root / f'L{FRAME_LAYERS[0]}').glob('*.pt'))).stem
print(f'spot-check stem={any_stem}')
for L in FRAME_LAYERS:
    t = torch.load(frames_root / f'L{L}' / f'{any_stem}.pt', map_location='cpu', weights_only=True)
    print(f'  L{L:02d}  shape={tuple(t.shape)}  dtype={t.dtype}')

SyntaxError: unterminated f-string literal (detected at line 49) (2223583577.py, line 49)

## A3 — Phoneme CTC labels (wav2vec2-xlsr-53-espeak-cv-ft)

Per-frame argmax IPA phoneme IDs at 50 Hz (same stride as WavLM). Writes
`cache/phoneme_labels/{stem}.pt` (int16), plus `vocab.json` and `categories.json`
(IPA → 6 categories: silence / vowel / nasal / fricative / plosive / approximant).

Spot-check asserts frame counts match the WavLM frame cache for the same stem
(must be equal or off-by-at-most-1 for the per-category pooling to index cleanly).

In [1]:
# A3 phoneme extraction - self-contained, idempotent.
import sys, json, torch
from pathlib import Path
from collections import Counter

MODEL_ROOT = Path.cwd() if Path.cwd().name == 'model' else Path('../model')
if str(MODEL_ROOT.resolve()) not in sys.path:
    sys.path.insert(0, str(MODEL_ROOT.resolve()))

from features.phoneme import extract_phonemes, PHONEME_CATEGORIES
from data.data import AudioDataset

DATA_DIR   = '../dataset/ComParE2017_Cold_4students'
CACHE_ROOT = '../cache'
BACKBONE   = 'wavlm-large'
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
CLIP_SECS  = 8.0
BATCH      = 4

for split in ('train', 'devel'):
    ds = AudioDataset(
        data_dir=DATA_DIR, split=split,
        use_mel=False, use_opensmile=False,
        pad_or_truncate_secs=CLIP_SECS,
    )
    print(f'phonemes[{split}] n={len(ds)}')
    report = extract_phonemes(
        dataset=ds,
        cache_root=CACHE_ROOT,
        device=DEVICE,
        batch_size=BATCH,
        skip_existing=True,
    )
    print(f'  wrote {report["n_written"]} new files  vocab_size={report["vocab_size"]}')

if DEVICE == 'cuda':
    torch.cuda.empty_cache()

# --- spot-check: phoneme frame count == WavLM frame count ---
labels_dir = Path(CACHE_ROOT) / 'phoneme_labels'
frames_dir = Path(CACHE_ROOT) / f'microsoft_{BACKBONE}' / 'frames' / 'L24'

stems = sorted([p.stem for p in labels_dir.glob('*.pt')])[:5]
print(f'spot-check (first 5 stems)')
for s in stems:
    lab = torch.load(labels_dir / f'{s}.pt', map_location='cpu', weights_only=True)
    frm = torch.load(frames_dir / f'{s}.pt',  map_location='cpu', weights_only=True)
    match = 'OK' if lab.shape[0] == frm.shape[0] else f'DIFF ({lab.shape[0] - frm.shape[0]:+d})'
    print(f'  {s}  labels={tuple(lab.shape)} dtype={lab.dtype}  frames={tuple(frm.shape)}  {match}')

# --- category histogram on a sample stem ---
cat_map = json.loads((labels_dir / 'categories.json').read_text(encoding='utf-8'))
id2cat = cat_map['id_to_category']
sample_lab = torch.load(labels_dir / f'{stems[0]}.pt', map_location='cpu', weights_only=True)
cats = [id2cat[int(x)] for x in sample_lab.tolist()]
hist = Counter(cats)
print(f'category histogram for {stems[0]} (T={len(cats)})')
for i, name in enumerate(PHONEME_CATEGORIES):
    print(f'  {i} {name:<12s} {hist.get(i, 0):4d}  ({hist.get(i, 0)/len(cats):.1%})')

phonemes[train] n=9505


phonemes[wav2vec2-xlsr-53-espeak-cv-ft]:   0%|          | 0/2377 [00:00<?, ?it/s]

  wrote 9505 new files  vocab_size=392
phonemes[devel] n=9596


phonemes[wav2vec2-xlsr-53-espeak-cv-ft]:   0%|          | 0/2399 [00:00<?, ?it/s]

  wrote 9596 new files  vocab_size=392
spot-check (first 5 stems)
  devel_0001  labels=(399,) dtype=torch.int16  frames=(399, 1024)  OK
  devel_0002  labels=(399,) dtype=torch.int16  frames=(399, 1024)  OK
  devel_0003  labels=(399,) dtype=torch.int16  frames=(399, 1024)  OK
  devel_0004  labels=(399,) dtype=torch.int16  frames=(399, 1024)  OK
  devel_0005  labels=(399,) dtype=torch.int16  frames=(399, 1024)  OK
category histogram for devel_0001 (T=399)
  0 silence         4  (1.0%)
  1 vowel          48  (12.0%)
  2 nasal          46  (11.5%)
  3 fricative      38  (9.5%)
  4 plosive       182  (45.6%)
  5 approximant    81  (20.3%)


## Diagnose Histogram Mismatch

In [2]:
from collections import Counter
import torch, json
from pathlib import Path

labels_dir = Path('../cache/phoneme_labels')
id2tok = json.loads((labels_dir / 'vocab.json').read_text(encoding='utf-8'))
lab = torch.load(labels_dir / 'devel_0001.pt', map_location='cpu', weights_only=True)
cnt = Counter(int(x) for x in lab.tolist())
print(f'top-10 raw tokens for devel_0001 (T={len(lab)})')
for tid, n in cnt.most_common(10):
    print(f'  id={tid:4d}  tok={id2tok[str(tid)]!r:8s}  n={n:3d}  ({n/len(lab):.1%})')


top-10 raw tokens for devel_0001 (T=399)
  id=   6  tok='t'       n=147  (36.8%)
  id=   4  tok='n'       n= 39  (9.8%)
  id=  15  tok='ɾ'       n= 31  (7.8%)
  id=  24  tok='j'       n= 29  (7.3%)
  id=   8  tok='l'       n= 14  (3.5%)
  id=   7  tok='ə'       n= 13  (3.3%)
  id=  12  tok='d'       n= 13  (3.3%)
  id=   5  tok='s'       n= 12  (3.0%)
  id=  14  tok='ɛ'       n=  8  (2.0%)
  id=   9  tok='a'       n=  8  (2.0%)


## A3 — Phoneme CTC labels (wav2vec2-xlsr-53-espeak-cv-ft)

Per-frame argmax IPA phoneme IDs at 50 Hz (same stride as WavLM). Writes
`cache/phoneme_labels/{stem}.pt` (int16), plus `vocab.json` and `categories.json`
(IPA → 6 categories: silence / vowel / nasal / fricative / plosive / approximant).

Spot-check asserts frame counts match the WavLM frame cache for the same stem
(must be equal or off-by-at-most-1 for the per-category pooling to index cleanly).

In [1]:
# A3 phoneme extraction - self-contained, idempotent.
import sys, json, torch
from pathlib import Path
from collections import Counter

MODEL_ROOT = Path.cwd() if Path.cwd().name == 'model' else Path('../model')
if str(MODEL_ROOT.resolve()) not in sys.path:
    sys.path.insert(0, str(MODEL_ROOT.resolve()))

from features.phoneme import extract_phonemes, PHONEME_CATEGORIES
from data.data import AudioDataset

DATA_DIR   = '../dataset/ComParE2017_Cold_4students'
CACHE_ROOT = '../cache'
BACKBONE   = 'wavlm-large'
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
CLIP_SECS  = 8.0
BATCH      = 4

for split in ('train', 'devel'):
    ds = AudioDataset(
        data_dir=DATA_DIR, split=split,
        use_mel=False, use_opensmile=False,
        pad_or_truncate_secs=CLIP_SECS,
    )
    print(f'phonemes[{split}] n={len(ds)}')
    report = extract_phonemes(
        dataset=ds,
        cache_root=CACHE_ROOT,
        device=DEVICE,
        batch_size=BATCH,
        skip_existing=True,
    )
    print(f'  wrote {report["n_written"]} new files  vocab_size={report["vocab_size"]}')

if DEVICE == 'cuda':
    torch.cuda.empty_cache()

# --- spot-check: phoneme frame count == WavLM frame count ---
labels_dir = Path(CACHE_ROOT) / 'phoneme_labels'
frames_dir = Path(CACHE_ROOT) / f'microsoft_{BACKBONE}' / 'frames' / 'L24'

stems = sorted([p.stem for p in labels_dir.glob('*.pt')])[:5]
print(f'spot-check (first 5 stems)')
for s in stems:
    lab = torch.load(labels_dir / f'{s}.pt', map_location='cpu', weights_only=True)
    frm = torch.load(frames_dir / f'{s}.pt',  map_location='cpu', weights_only=True)
    match = 'OK' if lab.shape[0] == frm.shape[0] else f'DIFF ({lab.shape[0] - frm.shape[0]:+d})'
    print(f'  {s}  labels={tuple(lab.shape)} dtype={lab.dtype}  frames={tuple(frm.shape)}  {match}')

# --- category histogram on a sample stem ---
cat_map = json.loads((labels_dir / 'categories.json').read_text(encoding='utf-8'))
id2cat = cat_map['id_to_category']
sample_lab = torch.load(labels_dir / f'{stems[0]}.pt', map_location='cpu', weights_only=True)
cats = [id2cat[int(x)] for x in sample_lab.tolist()]
hist = Counter(cats)
print(f'category histogram for {stems[0]} (T={len(cats)})')
for i, name in enumerate(PHONEME_CATEGORIES):
    print(f'  {i} {name:<12s} {hist.get(i, 0):4d}  ({hist.get(i, 0)/len(cats):.1%})')

phonemes[train] n=9505


phonemes[wav2vec2-xlsr-53-espeak-cv-ft]:   0%|          | 0/2377 [00:00<?, ?it/s]

  wrote 9505 new files  vocab_size=392
phonemes[devel] n=9596


phonemes[wav2vec2-xlsr-53-espeak-cv-ft]:   0%|          | 0/2399 [00:00<?, ?it/s]

  wrote 9596 new files  vocab_size=392
spot-check (first 5 stems)
  devel_0001  labels=(399,) dtype=torch.int16  frames=(399, 1024)  OK
  devel_0002  labels=(399,) dtype=torch.int16  frames=(399, 1024)  OK
  devel_0003  labels=(399,) dtype=torch.int16  frames=(399, 1024)  OK
  devel_0004  labels=(399,) dtype=torch.int16  frames=(399, 1024)  OK
  devel_0005  labels=(399,) dtype=torch.int16  frames=(399, 1024)  OK
category histogram for devel_0001 (T=399)
  0 silence       301  (75.4%)
  1 vowel          34  (8.5%)
  2 nasal          14  (3.5%)
  3 fricative      16  (4.0%)
  4 plosive        21  (5.3%)
  5 approximant    13  (3.3%)


## A3 — Phoneme CTC soft-aggregation diagnostic

Hard-argmax gave 75% blank. After masking blank, the runner-up was filler tokens (`t`, `ɾ`, `j`). Open question: is the model uniformly confused, or is real phonological signal present but smeared across non-top-1 mass?

This cell runs the model on a few devel stems and computes:
- **hard-argmax category histogram** — what `phoneme.py` writes today
- **soft-sum category histogram** — `softmax @ [V → 6 category projection]`, summed over time
- **conditional view** — on frames where blank wins top-1, where does the *non-silence* soft mass go?
- **confidence diagnostics** — mean top-1 prob, mean blank prob, mean per-frame entropy

If the soft histogram looks phonologically plausible (silence dominant but vowel/fricative/nasal each non-trivial) AND mean top-1 ≪ 1.0 (model is spreading mass), soft pooling is viable. If the soft histogram still collapses to silence + plosive, the model genuinely doesn't know and we fall back to pYIN+RMS.

In [1]:
# A3 phoneme soft-aggregation DIAGNOSTIC (no cache writes).
# Goal: decide whether to keep phoneme path (via soft category mass) or pivot to pYIN+RMS.
import sys, json, torch
from pathlib import Path

import numpy as np
from huggingface_hub import hf_hub_download
from transformers import Wav2Vec2ForCTC

MODEL_ROOT = Path.cwd() if Path.cwd().name == 'model' else Path('../model')
if str(MODEL_ROOT.resolve()) not in sys.path:
    sys.path.insert(0, str(MODEL_ROOT.resolve()))

from features.phoneme import build_category_map, PHONEME_CATEGORIES
from features.extract import _pad_collate
from data.data import AudioDataset
from torch.utils.data import Subset, DataLoader

DATA_DIR   = '../dataset/ComParE2017_Cold_4students'
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_NAME = 'facebook/wav2vec2-xlsr-53-espeak-cv-ft'
N_STEMS    = 8
CLIP_SECS  = 8.0

# ---- vocab + category projection (same logic as phoneme.py) ----
vocab_path = hf_hub_download(repo_id=MODEL_NAME, filename='vocab.json')
vocab  = json.loads(Path(vocab_path).read_text(encoding='utf-8'))
id2tok = {v: k for k, v in vocab.items()}
cat_map = build_category_map(vocab)
id2cat  = np.asarray(cat_map['id_to_category'])      # [V] int
V       = len(id2cat)
NCAT    = len(PHONEME_CATEGORIES)
BLANK_ID = 0                                          # CTC blank, conventional

# [V, 6] one-hot projection into IPA categories
proj = np.zeros((V, NCAT), dtype=np.float32)
proj[np.arange(V), id2cat] = 1.0
proj_t = torch.from_numpy(proj).to(DEVICE)

# Mask of non-silence vocab ids (drops blank, special tokens, anything else
# build_category_map() bucketed as silence). Used for the conditional view.
nonsilent_mask = id2cat != 0
proj_ns = proj.copy(); proj_ns[~nonsilent_mask, :] = 0.0
proj_ns_t = torch.from_numpy(proj_ns).to(DEVICE)

print(f'vocab V={V}  non-silence ids={int(nonsilent_mask.sum())}/{V}')

# ---- data ----
ds_full = AudioDataset(
    data_dir=DATA_DIR, split='devel',
    use_mel=False, use_opensmile=False,
    pad_or_truncate_secs=CLIP_SECS,
)
ds = Subset(ds_full, list(range(N_STEMS)))
loader = DataLoader(ds, batch_size=N_STEMS, shuffle=False, collate_fn=_pad_collate)
batch = next(iter(loader))

# ---- model + forward ----
model = (Wav2Vec2ForCTC
         .from_pretrained(MODEL_NAME, torch_dtype=torch.float16)
         .eval()
         .to(DEVICE))

audio = batch['audio'].to(DEVICE, dtype=torch.float32)
attn  = batch['attention_mask'].to(DEVICE, dtype=torch.long)

# Per-sample fp32 mean/var normalisation over valid frames (matches phoneme.py)
m = attn.to(torch.float32)
n = m.sum(-1, keepdim=True).clamp(min=1.0)
mean = (audio * m).sum(-1, keepdim=True) / n
centered = (audio - mean) * m
var = (centered ** 2).sum(-1, keepdim=True) / n
audio_norm = ((audio - mean) / (var + 1e-7).sqrt()) * m
audio_norm = audio_norm.to(torch.float16)

with torch.no_grad():
    logits = model(input_values=audio_norm, attention_mask=attn).logits  # [B, T, V]
    probs  = logits.float().softmax(dim=-1)                                # [B, T, V]

in_lens  = attn.sum(-1)
out_lens = model.wav2vec2._get_feat_extract_output_lengths(in_lens).long()

# ---- per-stem comparison: hard-argmax vs soft-sum ----
print(f'\n== A3 phoneme diagnostic on {N_STEMS} devel stems ==')
print(f'   model: {MODEL_NAME}\n')

per_stem_soft, per_stem_hard = [], []
blank_fracs, mean_top1, mean_blank_p, mean_entropy = [], [], [], []

for i in range(N_STEMS):
    Tv = int(out_lens[i].item())
    p  = probs[i, :Tv]                              # [Tv, V]

    # soft category mass: sum probs by IPA category, then normalise over time
    cat_p_total = (p @ proj_t).sum(dim=0).cpu().numpy()
    soft = cat_p_total / cat_p_total.sum()
    per_stem_soft.append(soft)

    # hard-argmax category histogram
    pred = p.argmax(dim=-1).cpu().numpy()
    hard = np.zeros(NCAT)
    for c in range(NCAT):
        hard[c] = float((id2cat[pred] == c).sum())
    hard /= hard.sum()
    per_stem_hard.append(hard)

    # confidence proxies
    blank_fracs.append(float((pred == BLANK_ID).mean()))
    mean_top1.append(float(p.max(dim=-1).values.mean().item()))
    mean_blank_p.append(float(p[:, BLANK_ID].mean().item()))
    H = -(p * (p.clamp_min(1e-12)).log()).sum(dim=-1)   # [Tv]
    mean_entropy.append(float(H.mean().item()))

    stem = batch['file_name'][i]
    if stem.endswith('.wav'): stem = stem[:-4]
    print(f'-- {stem} (T={Tv}) --')
    print(f'   blank-wins-top1: {blank_fracs[-1]:.1%}   '
          f'mean(top1 prob): {mean_top1[-1]:.3f}   '
          f'mean(blank prob): {mean_blank_p[-1]:.3f}   '
          f'mean(entropy): {mean_entropy[-1]:.2f}  (uniform = {np.log(V):.2f})')
    print(f'   {"category":<12s}  {"hard-argmax":>11s}  {"soft-sum":>11s}')
    for c, name in enumerate(PHONEME_CATEGORIES):
        print(f'   {name:<12s}  {hard[c]:>10.1%}  {soft[c]:>10.1%}')
    print()

# ---- aggregate across stems ----
soft_mean = np.mean(per_stem_soft, axis=0)
hard_mean = np.mean(per_stem_hard, axis=0)
print(f'== AGGREGATE over {N_STEMS} stems ==')
print(f'   mean blank-wins-top1: {np.mean(blank_fracs):.1%}')
print(f'   mean top-1 prob:      {np.mean(mean_top1):.3f}    (1.0 = certain, 1/V ≈ {1/V:.4f} = uniform)')
print(f'   mean blank prob:      {np.mean(mean_blank_p):.3f}')
print(f'   mean entropy (nats):  {np.mean(mean_entropy):.2f}    (uniform over V ≈ {np.log(V):.2f})')
print(f'   {"category":<12s}  {"hard-argmax":>11s}  {"soft-sum":>11s}')
for c, name in enumerate(PHONEME_CATEGORIES):
    print(f'   {name:<12s}  {hard_mean[c]:>10.1%}  {soft_mean[c]:>10.1%}')

# ---- conditional: in blank-winning frames, where does NON-silence soft mass go? ----
print(f'\n== CONDITIONAL on blank-winning frames: distribution of NON-silence soft mass ==')
cond_total = np.zeros(NCAT); n_blank_frames = 0
for i in range(N_STEMS):
    Tv = int(out_lens[i].item())
    p = probs[i, :Tv]
    blank_idx = (p.argmax(dim=-1) == BLANK_ID)
    if not bool(blank_idx.any()): continue
    p_blank = p[blank_idx]                                 # [Bb, V]
    cond_total += (p_blank @ proj_ns_t).sum(dim=0).cpu().numpy()
    n_blank_frames += int(blank_idx.sum().item())

if cond_total.sum() > 0:
    cond_norm = cond_total / cond_total.sum()
    print(f'   ({n_blank_frames} blank-winning frames, non-silence mass redistributed)')
    print(f'   {"category":<12s}  {"share":>8s}')
    for c, name in enumerate(PHONEME_CATEGORIES):
        if c == 0: continue
        print(f'   {name:<12s}  {cond_norm[c]:>7.1%}')

del model
if DEVICE == 'cuda': torch.cuda.empty_cache()

vocab V=392  non-silence ids=371/392

== A3 phoneme diagnostic on 8 devel stems ==
   model: facebook/wav2vec2-xlsr-53-espeak-cv-ft

-- devel_0001 (T=399) --
   blank-wins-top1: 74.7%   mean(top1 prob): 0.953   mean(blank prob): 0.748   mean(entropy): 0.20  (uniform = 5.97)
   category      hard-argmax     soft-sum
   silence            75.4%       75.6%
   vowel               8.5%        8.3%
   nasal               3.5%        3.8%
   fricative           4.0%        4.0%
   plosive             5.3%        5.4%
   approximant         3.3%        2.9%

-- devel_0002 (T=399) --
   blank-wins-top1: 92.0%   mean(top1 prob): 0.984   mean(blank prob): 0.915   mean(entropy): 0.07  (uniform = 5.97)
   category      hard-argmax     soft-sum
   silence            92.0%       91.5%
   vowel               2.8%        2.6%
   nasal               1.3%        1.4%
   fricative           2.0%        2.1%
   plosive             1.8%        1.8%
   approximant         0.3%        0.6%

-- devel_0003 (T=

## A3 (revised) — Acoustic manner labels (pYIN voicing + RMS energy)

Replaces the phoneme-CTC path. 3 categories (silence / voiced / unvoiced)
aligned to the WavLM frame cache at 50 Hz. pYIN provides voiced/unvoiced
flags with standard validation in the speech literature; smearing
heuristics on underconfident CTC output cannot be validated against any
ground truth we have for URTIC.

**Validation gate (time-boxed)**: run on 20 sample chunks first. If the
histogram is sane (silence 20–40%, voiced 45–65%, unvoiced 10–25%) and
the voiced time-ranges land on what look like vowel/nasal regions of the
waveform, proceed to full extraction. Otherwise skip A3 → A4.

In [1]:
# A3 manner labels - VALIDATION ONLY on 20 sample chunks.
import sys, json, torch, numpy as np
from pathlib import Path
from collections import Counter

MODEL_ROOT = Path.cwd() if Path.cwd().name == 'model' else Path('../model')
if str(MODEL_ROOT.resolve()) not in sys.path:
    sys.path.insert(0, str(MODEL_ROOT.resolve()))

from features.manner import extract_manner_labels, MANNER_CATEGORIES, compute_manner
from data.data import AudioDataset
from torch.utils.data import Subset

DATA_DIR   = '../dataset/ComParE2017_Cold_4students'
CACHE_ROOT = '../cache'
BACKBONE   = 'wavlm-large'
CLIP_SECS  = 8.0

ds_full = AudioDataset(
    data_dir=DATA_DIR, split='devel',
    use_mel=False, use_opensmile=False,
    pad_or_truncate_secs=CLIP_SECS,
)
ds = Subset(ds_full, list(range(20)))

# Use a tmp subdir so validation labels don't pollute the full cache
TMP_ROOT = '../cache/_manner_validate'
report = extract_manner_labels(
    dataset=ds, cache_root=TMP_ROOT,
    frames_cache_root=CACHE_ROOT,   # read WavLM frame counts from the real cache
    backbone_id=f'microsoft_{BACKBONE}',
    skip_existing=False,
)
print(f'wrote {report["n_written"]} validation labels → {report["out_dir"]}')

# --- aggregate histogram over the 20 chunks ---
labels_dir = Path(TMP_ROOT) / 'manner_labels'
counts = Counter()
totals = 0
for p in sorted(labels_dir.glob('*.pt'))[:20]:
    lab = torch.load(p, map_location='cpu', weights_only=True).numpy()
    for c in range(len(MANNER_CATEGORIES)):
        counts[c] += int((lab == c).sum())
    totals += lab.shape[0]

print(f'aggregate histogram over 20 chunks (total frames = {totals})')
for c, name in enumerate(MANNER_CATEGORIES):
    frac = counts[c] / totals
    print(f'  {c} {name:<9s} {counts[c]:5d}  ({frac:.1%})')

# --- time-range dump for one stem, for waveform spot-check ---
sample = sorted(labels_dir.glob('*.pt'))[0]
lab = torch.load(sample, map_location='cpu', weights_only=True).numpy()
hop_s = 320 / 16000  # 20 ms
segments = []
cur_cat, cur_start = int(lab[0]), 0
for i in range(1, len(lab)):
    if int(lab[i]) != cur_cat:
        segments.append((cur_cat, cur_start, i))
        cur_cat, cur_start = int(lab[i]), i
segments.append((cur_cat, cur_start, len(lab)))

print(f'time-ranges for {sample.stem} (T={len(lab)} frames = {len(lab)*hop_s:.2f}s)')
print('  category    t_start    t_end    dur (ms)')
for c, a, b in segments[:30]:
    print(f'  {MANNER_CATEGORIES[c]:<9s}  {a*hop_s:6.3f}s  {b*hop_s:6.3f}s  {(b-a)*hop_s*1000:5.0f}')
if len(segments) > 30:
    print(f'  ... ({len(segments) - 30} more segments)')

manner[pYIN+RMS]:   0%|          | 0/20 [00:00<?, ?it/s]

wrote 20 validation labels → ..\cache\_manner_validate\manner_labels
aggregate histogram over 20 chunks (total frames = 7980)
  0 silence    3226  (40.4%)
  1 voiced     3010  (37.7%)
  2 unvoiced   1744  (21.9%)
time-ranges for devel_0001 (T=399 frames = 7.98s)
  category    t_start    t_end    dur (ms)
  silence     0.000s   0.080s     80
  unvoiced    0.080s   0.180s    100
  voiced      0.180s   0.760s    580
  unvoiced    0.760s   1.320s    560
  voiced      1.320s   1.680s    360
  unvoiced    1.680s   1.980s    300
  voiced      1.980s   2.400s    420
  unvoiced    2.400s   2.600s    200
  voiced      2.600s   3.380s    780
  unvoiced    3.380s   3.460s     80
  voiced      3.460s   3.780s    320
  unvoiced    3.780s   3.800s     20
  voiced      3.800s   4.360s    560
  unvoiced    4.360s   4.420s     60
  voiced      4.420s   5.240s    820
  unvoiced    5.240s   5.580s    340
  silence     5.580s   5.600s     20
  unvoiced    5.600s   5.640s     40
  voiced      5.640s   6.520

In [ ]:
# Full extraction over train + devel — RUN ONLY IF VALIDATION HISTOGRAM PASSED.
import sys, torch
from pathlib import Path

MODEL_ROOT = Path.cwd() if Path.cwd().name == 'model' else Path('../model')
if str(MODEL_ROOT.resolve()) not in sys.path:
    sys.path.insert(0, str(MODEL_ROOT.resolve()))

from features.manner import extract_manner_labels
from data.data import AudioDataset

DATA_DIR   = '../dataset/ComParE2017_Cold_4students'
CACHE_ROOT = '../cache'
BACKBONE   = 'wavlm-large'
CLIP_SECS  = 8.0

for split in ('train', 'devel'):
    ds = AudioDataset(
        data_dir=DATA_DIR, split=split,
        use_mel=False, use_opensmile=False,
        pad_or_truncate_secs=CLIP_SECS,
    )
    print(f'manner[{split}] n={len(ds)}')
    report = extract_manner_labels(
        dataset=ds, cache_root=CACHE_ROOT,
        backbone_id=f'microsoft_{BACKBONE}',
        skip_existing=True,
    )
    print(f'  wrote {report["n_written"]} new labels')